## 1. Estado Objetivo e Exibição do Tabuleiro

In [49]:
OBJETIVO = (1, 2, 3, 4, 5, 6, 7, 8, 'X')  # Configuração-meta do 8-Puzzle a ser alcançada pelas buscas

def imprimir_tabuleiro(estado):  # Representação visual da grade 3x3 para inspeção do trajeto
    """Imprime o tabuleiro 3x3 no formato solicitado pelo professor."""
    print(f"|{estado[0]} | {estado[1]} | {estado[2]}|")  # Renderiza a primeira linha da matriz do jogo
    print(f"|{estado[3]} | {estado[4]} | {estado[5]}|")  # Renderiza a segunda linha da matriz do jogo
    print(f"|{estado[6]} | {estado[7]} | {estado[8]}|")  # Renderiza a terceira linha da matriz do jogo

print("Estado Objetivo:")  # Identifica o estado-alvo da busca
imprimir_tabuleiro(OBJETIVO)  # Exibe o arranjo final esperado das peças e do espaço vazio 'X'


Estado Objetivo:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|


## 2. Verificação de Solubilidade


In [50]:
def contar_inversoes(estado):  # Calcula a desordem do tabuleiro para provar algebricamente se a solução existe
    """Conta quantos números maiores aparecem antes de números menores."""
    numeros = [x for x in estado if x != 'X']  # Isola as peças numéricas, pois o espaço vazio 'X' não conta como inversão
    inversoes = 0  # Inicia a contagem de pares de peças em ordem decrescente relativa
    for i in range(len(numeros)):  # Fixa uma peça para comparar com as posições seguintes no arranjo
        for j in range(i + 1, len(numeros)):  # Avalia cada peça posterior em relação à fixada
            if numeros[i] > numeros[j]:  # Detecta inversão quando uma peça maior antecede uma menor
                inversoes += 1  # Contabiliza mais um desvio em relação à ordenação do objetivo
    return inversoes  # Total de permutações necessárias para ordenar o arranjo numérico

def eh_soluvel(estado):  # Teorema do 8-Puzzle: o espaço de estados divide-se em dois subgrafos desconexos
    """Retorna True se o número de inversões for par."""
    return contar_inversoes(estado) % 2 == 0  # No tabuleiro 3x3, apenas estados com paridade par de inversões alcançam o objetivo

tabuleiro_par = (1, 2, 3, 4, 5, 6, 7, 'X', 8)  # Cenário de teste pertencente ao subgrafo solúvel (alcançável)
tabuleiro_impar = (2, 1, 3, 4, 5, 6, 7, 8, 'X')  # Cenário de teste pertencente ao subgrafo insolúvel (impossível de resolver)

print(f"Teste 1: {contar_inversoes(tabuleiro_par)} inversões -> Solúvel? {eh_soluvel(tabuleiro_par)}")  # Valida que paridade par garante existência de solução
print(f"Teste 2: {contar_inversoes(tabuleiro_impar)} inversão -> Solúvel? {eh_soluvel(tabuleiro_impar)}")  # Valida que paridade ímpar impede a resolução por busca


Teste 1: 0 inversões -> Solúvel? True
Teste 2: 1 inversão -> Solúvel? False


## 3. Geração do Estado Inicial Aleatório


In [51]:
import random  # Gera permutações estocásticas do espaço de estados do quebra-cabeça

def gerar_estado_inicial():  # Amostra um ponto de partida válido e solúvel no espaço de busca
    """Gera um estado inicial aleatório e garante que ele seja solúvel."""
    pecas = [1, 2, 3, 4, 5, 6, 7, 8, 'X']  # Conjunto completo dos elementos móveis do tabuleiro
    tentativa = 1  # Mede o número de sorteios até obter um estado no subgrafo viável
    while True:  # Repete a amostragem até satisfazer os critérios matemáticos de solvência
        random.shuffle(pecas)  # Produz uma permutação aleatória dentre as 9!/2 configurações possíveis
        estado = tuple(pecas)  # Fixa a configuração do estado para identificação única na árvore de busca
        inversoes = contar_inversoes(estado)  # Avalia a paridade do estado gerado aleatoriamente
        
        if eh_soluvel(estado) and estado != OBJETIVO:  # Garante que o estado tem solução e não inicia já resolvido
            print(f"Tentativa {tentativa}: Estado solúvel aceito! (Inversões: {inversoes} - Par)")  # Confirma que o estado pertence ao mesmo espaço conexo do objetivo
            return estado  # Retorna o nó-raiz válido para alimentar os algoritmos de busca
        else:  # Filtra configurações insolúveis (50% de probabilidade matemática) ou triviais
            print(f"Tentativa {tentativa}: Descartado (Inversões: {inversoes} - Ímpar, gerando outro...)")  # Evita loops infinitos de busca em estados matematicamente impossíveis
            tentativa += 1  # Registra tentativas rejeitadas pelo teste de solubilidade

random.seed(15)  # Fixa o estado inicial para comparação justa de iterações entre BFS e DFS

estado_inicial = gerar_estado_inicial()  # Define o problema de busca que ambos os algoritmos tentarão resolver
print("\nEstado Inicial:")  # Sinaliza a configuração de partida da busca
imprimir_tabuleiro(estado_inicial)  # Visualiza a desordem inicial que o agente de busca precisa ordenar


Tentativa 1: Descartado (Inversões: 17 - Ímpar, gerando outro...)
Tentativa 2: Estado solúvel aceito! (Inversões: 12 - Par)

Estado Inicial:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|


## 4. Movimentos Válidos (Sucessores)

In [52]:
def gerar_sucessores(estado):  # Modelo de transição do ambiente: gera os estados vizinhos válidos
    """Gera os próximos estados possíveis movimentando o 'X'."""
    sucessores = []  # Conjunto de nós filhos gerados a partir do nó corrente
    idx = estado.index('X')  # Localiza a peça vazia, cujo deslocamento define as ações no jogo
    linha = idx // 3  # Mapeia a coordenada vertical de 'X' na grade bidimensional
    coluna = idx % 3  # Mapeia a coordenada horizontal de 'X' na grade bidimensional

    movimentos = [  # Espaço de ações possíveis do agente no tabuleiro
        (-1, 0),  # Ação de mover 'X' para cima (deslocamento vertical -1)
        (1, 0),   # Ação de mover 'X' para baixo (deslocamento vertical +1)
        (0, -1),  # Ação de mover 'X' para a esquerda (deslocamento horizontal -1)
        (0, 1)    # Ação de mover 'X' para a direita (deslocamento horizontal +1)
    ]

    for dl, dc in movimentos:  # Avalia a viabilidade física de cada ação elementar
        nova_l = linha + dl  # Posição vertical pretendida para a nova casa de 'X'
        nova_c = coluna + dc  # Posição horizontal pretendida para a nova casa de 'X'
        
        if 0 <= nova_l < 3 and 0 <= nova_c < 3:  # Restrição de contorno: descarta movimentos que sairiam dos limites da grade
            novo_idx = nova_l * 3 + nova_c  # Endereço da peça vizinha que trocará de lugar com 'X'
            novo_estado = list(estado)  # Clona a configuração para aplicar a transição sem corromper o estado pai
            novo_estado[idx], novo_estado[novo_idx] = novo_estado[novo_idx], novo_estado[idx]  # Aplica a transição: permuta 'X' com a peça adjacente
            sucessores.append(tuple(novo_estado))  # Insere o novo nó filho na lista de ramificação

    return sucessores  # Retorna o fator de ramificação do nó atual (de 2 a 4 sucessores válidos)


## 5. Busca em Largura (BFS)


In [53]:
from collections import deque  # Estrutura para implementar a fronteira FIFO (First-In, First-Out) da BFS

def busca_largura(estado_inicial, max_impressoes=10):  # Busca em Largura: explora o grafo nível por nível garantindo caminho ótimo
    """Executa a Busca em Largura (BFS)."""
    print("=" * 45)  # Delimita o início do experimento com BFS
    print("BUSCA EM LARGURA (BFS)")  # Identifica o algoritmo de busca cega ótima em teste
    print("=" * 45)  # Organização visual da execução
    print("Estado inicial:")  # Mostra a raiz da árvore de busca em largura
    imprimir_tabuleiro(estado_inicial)  # Configuração inicial a partir da qual a BFS buscará a menor sequência de passos
    print()  # Separação visual do rastreamento de passos

    fila = deque([estado_inicial])  # Fronteira de busca: garante que nós em profundidades menores sejam expandidos antes
    visitados = {estado_inicial}  # Lista de fechados (Closed List): poda ciclos e estados repetidos no grafo
    iteracoes = 0  # Métrica de custo computacional N: conta o total de nós expandidos até a meta

    while fila:  # Executa a busca enquanto houver estados pendentes na fronteira
        iteracoes += 1  # Contabiliza a expansão de mais um nó no espaço de busca
        atual = fila.popleft()  # Seleciona o nó mais raso disponível na fronteira (estratégia breadth-first)

        if iteracoes <= max_impressoes:  # Limita o rastreamento em tela para ilustrar o avanço sem poluir a saída
            print(f"Iteração {iteracoes}:")  # Identificador da iteração no rastreamento
            imprimir_tabuleiro(atual)  # Exibe a configuração intermediária avaliada nesta etapa
            print()  # Espaçamento de leitura
        elif iteracoes == max_impressoes + 1:  # Transição do modo detalhado para o modo de processamento silencioso
            print("...\n")  # Indica continuidade da busca em larga escala até atingir a meta

        if atual == OBJETIVO:  # Teste de meta: verifica se o nó atual atingiu a configuração-alvo
            print("Estado Final:")  # Confirma que a meta foi encontrada na menor profundidade possível
            imprimir_tabuleiro(atual)  # Exibe o tabuleiro solucionado pelo algoritmo
            print(f"\nNúmero de iterações para busca em largura N: {iteracoes}")  # Métrica N do exercício para medir o custo de nós da BFS
            return iteracoes  # Retorna o esforço computacional (nós expandidos) para resolver o problema

        for vizinho in gerar_sucessores(atual):  # Expande os filhos do nó atual para aprofundar na árvore
            if vizinho not in visitados:  # Poda de redundância: evita reprocessar configurações já exploradas
                visitados.add(vizinho)  # Registra o nó no grafo fechado logo na geração para evitar duplicatas na fila
                fila.append(vizinho)  # Enfileira o filho no fim da fila para ser expandido após todos do nível atual

    return iteracoes  # Retorna iterações caso o espaço se esgote sem solução

n_bfs = busca_largura(estado_inicial)  # Executa a BFS completa e armazena N_BFS para comparação teórica


BUSCA EM LARGURA (BFS)
Estado inicial:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 1:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 2:
|8 | 2 | 4|
|3 | 1 | X|
|5 | 7 | 6|

Iteração 3:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | X | 7|

Iteração 4:
|8 | 2 | X|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 5:
|8 | 2 | 4|
|3 | X | 1|
|5 | 7 | 6|

Iteração 6:
|8 | 2 | 4|
|3 | X | 6|
|5 | 1 | 7|

Iteração 7:
|8 | 2 | 4|
|3 | 1 | 6|
|X | 5 | 7|

Iteração 8:
|8 | X | 2|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 9:
|8 | X | 4|
|3 | 2 | 1|
|5 | 7 | 6|

Iteração 10:
|8 | 2 | 4|
|3 | 7 | 1|
|5 | X | 6|

...

Estado Final:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Número de iterações para busca em largura N: 80950


## 6. Busca em Profundidade (DFS)


In [54]:
def busca_profundidade(estado_inicial, max_impressoes=10):  # Busca em Profundidade: prioriza aprofundamento contínuo em ramos antes do backtracking
    """Executa a Busca em Profundidade (DFS)."""
    print("=" * 45)  # Delimita o início do experimento com DFS
    print("BUSCA EM PROFUNDIDADE (DFS)")  # Identifica o algoritmo de busca cega não-ótima em teste
    print("=" * 45)  # Organização visual da execução
    print("Estado inicial:")  # Mostra a raiz da árvore de busca em profundidade
    imprimir_tabuleiro(estado_inicial)  # Configuração inicial a ser explorada em profundidade
    print()  # Separação visual do rastreamento de passos

    pilha = [estado_inicial]  # Fronteira LIFO (Last-In, First-Out): prioriza os nós gerados mais recentemente
    visitados = {estado_inicial}  # Lista de fechados: mecanismo crítico na DFS para não entrar em ciclos infinitos
    iteracoes = 0  # Métrica de custo computacional N: conta o total de nós expandidos no mergulho

    while pilha:  # Continua enquanto houver caminhos para aprofundar ou retroceder (backtrack)
        iteracoes += 1  # Contabiliza a exploração de mais um estado no ramo corrente
        atual = pilha.pop()  # Retira o nó mais profundo da fronteira para exploração imediata (estratégia depth-first)

        if iteracoes <= max_impressoes:  # Amostragem dos passos iniciais do mergulho em profundidade
            print(f"Iteração {iteracoes}:")  # Identificador da iteração no rastreamento
            imprimir_tabuleiro(atual)  # Exibe a configuração do tabuleiro neste ponto do ramo
            print()  # Espaçamento de leitura
        elif iteracoes == max_impressoes + 1:  # Transição para exploração silenciosa sem sobrecarregar a tela
            print("...\n")  # Indica continuidade dos mergulhos e retrocessos no grafo

        if atual == OBJETIVO:  # Teste de meta: verifica se o ramo explorado encontrou o objetivo (não garante menor caminho)
            print("Estado Final:")  # Confirma que a meta foi alcançada em algum ramo da árvore
            imprimir_tabuleiro(atual)  # Exibe o tabuleiro solucionado pelo algoritmo
            print(f"\nNúmero de iterações para busca em profundidade N: {iteracoes}")  # Métrica N do exercício para medir o custo de nós da DFS
            return iteracoes  # Retorna o esforço computacional (nós expandidos) até a solução pela DFS

        for vizinho in reversed(gerar_sucessores(atual)):  # Ordem reversa para que o primeiro filho gerado seja o primeiro aprofundado
            if vizinho not in visitados:  # Poda de ciclos: impede que a busca reexplore estados já visitados no ramo ou na árvore
                visitados.add(vizinho)  # Registra o nó visitado para bloquear reentradas cíclicas
                pilha.append(vizinho)  # Adiciona ao topo da pilha para mergulhar imediatamente nesse nó no próximo passo

    return iteracoes  # Retorna iterações caso o espaço de busca se esgote sem solução

n_dfs = busca_profundidade(estado_inicial)  # Executa a DFS completa e armazena N_DFS para comparação com a BFS


BUSCA EM PROFUNDIDADE (DFS)
Estado inicial:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 1:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 2:
|8 | 2 | 4|
|3 | 1 | X|
|5 | 7 | 6|

Iteração 3:
|8 | 2 | X|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 4:
|8 | X | 2|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 5:
|8 | 1 | 2|
|3 | X | 4|
|5 | 7 | 6|

Iteração 6:
|8 | 1 | 2|
|3 | 7 | 4|
|5 | X | 6|

Iteração 7:
|8 | 1 | 2|
|3 | 7 | 4|
|X | 5 | 6|

Iteração 8:
|8 | 1 | 2|
|X | 7 | 4|
|3 | 5 | 6|

Iteração 9:
|X | 1 | 2|
|8 | 7 | 4|
|3 | 5 | 6|

Iteração 10:
|1 | X | 2|
|8 | 7 | 4|
|3 | 5 | 6|

...

Estado Final:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Número de iterações para busca em profundidade N: 24098


## 7. Resumo Final das Iterações

In [55]:
print("=" * 45)  # Delimita o início do relatório comparativo
print("RESUMO FINAL:")  # Cabeçalho da comparação de desempenho entre os algoritmos de busca cega
print(f"Número de iterações para busca em largura N: {n_bfs}")  # Custo N da BFS: busca exaustiva por camadas garantindo solução ótima
print(f"Número de iterações para busca em profundidade N: {n_dfs}")  # Custo N da DFS: busca gulosa em profundidade com caminho geralmente subótimo
print("=" * 45)  # Encerramento do relatório de métricas solicitado pelo exercício


RESUMO FINAL:
Número de iterações para busca em largura N: 80950
Número de iterações para busca em profundidade N: 24098


### Demonstração

In [56]:
tabuleiro_curto = (1, 2, 3, 4, 'X', 6, 7, 5, 8)  # Estado de teste próximo da meta para validar didaticamente a árvore de busca

print("--- Teste Rápido com BFS ---")  # Rótulo de teste comparativo para a Busca em Largura
busca_largura(tabuleiro_curto, max_impressoes=20)  # Executa a BFS mostrando a descoberta da solução ótima em poucas transições

print("\n--- Teste Rápido com DFS ---")  # Rótulo de teste comparativo para a Busca em Profundidade
busca_profundidade(tabuleiro_curto, max_impressoes=20)  # Executa a DFS evidenciando o comportamento de desvio e mergulho profundo


--- Teste Rápido com BFS ---
BUSCA EM LARGURA (BFS)
Estado inicial:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 1:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 2:
|1 | X | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 3:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | X | 8|

Iteração 4:
|1 | 2 | 3|
|X | 4 | 6|
|7 | 5 | 8|

Iteração 5:
|1 | 2 | 3|
|4 | 6 | X|
|7 | 5 | 8|

Iteração 6:
|X | 1 | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 7:
|1 | 3 | X|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 8:
|1 | 2 | 3|
|4 | 5 | 6|
|X | 7 | 8|

Iteração 9:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Estado Final:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Número de iterações para busca em largura N: 9

--- Teste Rápido com DFS ---
BUSCA EM PROFUNDIDADE (DFS)
Estado inicial:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 1:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 2:
|1 | X | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 3:
|X | 1 | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 4:
|4 | 1 | 3|
|X | 2 | 6|
|7 | 5 | 8|

Iteração 5:
|4 | 1 | 3|
|7 | 2 | 6|
|X | 5 | 8|



329